# Rhyme tools
Pronunciations, syllable counts, a rhyme score and rhyme richness, all from CMUdict.

Steps:
1. load CMUdict
2. add the words it doesn't have
3. count syllables
4. rhyme score
5. rhyme richness
6. check it on real poems

Other notebooks can load all of this with `%run rhyme.ipynb` (takes a few seconds).

In [25]:
import re
import unicodedata
from collections import Counter, defaultdict
import pandas as pd

DATA = "../data/"

pron = defaultdict(list)
with open(DATA + "cmudict/cmudict-0.7b", encoding="latin-1") as f:
    for line in f:
        if line.startswith(";;;") or not line.strip():
            continue
        word, phones = line.strip().split("  ", 1)
        word = word.split("(")[0].lower()  # WORD(1) is a second way to say WORD
        if re.fullmatch(r"[a-z']+", word):
            pron[word].append(phones)
pron = dict(pron)

print(len(pron), "words")
print("light:", pron["light"])
print("wind:", pron["wind"])

124567 words
light: ['L AY1 T']
wind: ['W AY1 N D', 'W IH1 N D']


## 1. Words CMUdict doesn't have
CMUdict is modern American English, so a lot of old poetry words are missing. We add the most common
ones that end lines in our training data by hand (our own transcriptions, in CMUdict's style).

Two more rules catch a lot: `crown'd` is said like `crowned`, and plurals like `rills` are the word + s.

In [26]:
ARCHAIC = {
    "o'er": "AO1 R", "e'er": "EH1 R", "e'en": "IY1 N", "heav'n": "HH EH1 V N", "thro": "TH R UW1",
    "clime": "K L AY1 M", "jove": "JH OW1 V", "nought": "N AO1 T", "drear": "D R IH1 R",
    "aright": "AH0 R AY1 T", "saith": "S EH1 TH", "ween": "W IY1 N", "amain": "AH0 M EY1 N",
    "affright": "AH0 F R AY1 T", "withal": "W IH0 DH AO1 L", "drest": "D R EH1 S T",
    "offence": "AH0 F EH1 N S", "pretence": "P R IY0 T EH1 N S", "smite": "S M AY1 T",
    "sooth": "S UW1 TH", "hie": "HH AY1", "weal": "W IY1 L", "pelf": "P EH1 L F", "clod": "K L AA1 D",
    "repine": "R IH0 P AY1 N", "blent": "B L EH1 N T", "smote": "S M OW1 T", "eventide": "IY1 V AH0 N T AY2 D",
    "gladness": "G L AE1 D N AH0 S", "loveliness": "L AH1 V L IY0 N AH0 S", "firmament": "F ER1 M AH0 M AH0 N T",
}
for w, p in ARCHAIC.items():
    pron.setdefault(w, [p])


def phones(word):
    word = word.lower()
    if word in pron:
        return pron[word]
    if word.endswith("'d") and word[:-2] + "ed" in pron:  # crown'd -> crowned
        return pron[word[:-2] + "ed"]
    if word.endswith("s") and word[:-1] in pron:  # rills -> rill + s (roses -> IH0 Z, deeps -> S)
        ending = {"S": " IH0 Z", "Z": " IH0 Z", "SH": " IH0 Z", "CH": " IH0 Z", "JH": " IH0 Z",
                  "P": " S", "T": " S", "K": " S", "F": " S", "TH": " S"}
        return [p + ending.get(p.split()[-1], " Z") for p in pron[word[:-1]]]
    return []


def words_of(line):
    # o’er -> o'er, blessèd -> blessed (otherwise they get split in two)
    line = unicodedata.normalize("NFKD", line.lower().replace("’", "'")).encode("ascii", "ignore").decode()
    words = re.findall(r"[a-z']+", line)
    words = [w if phones(w) else w.strip("'") for w in words]
    return [w for w in words if w]


print(phones("crown'd"), phones("rills"), phones("roses"), phones("o'er"))
print(words_of("O’er the blessèd sea-shore,"))

['K R AW1 N D'] ['R IH1 L Z'] ['R OW1 Z IH0 Z'] ['AO1 R']
["o'er", 'the', 'blessed', 'sea', 'shore']


How many line-ending words do we still not know? Checked on the val poems. Roman numerals (II, III, ...)
are section headings, not words, so they don't count.

In [27]:
val = pd.read_parquet(DATA + "train_test/val.parquet", columns=["author","text"])


# def poem_lines(text):
#     poem = text.split("\n\n", 1)[1].replace("<|endoftext|>", "")
#     return [line for line in poem.split("\n") if line.strip()]

def poem_lines(text):
    if not isinstance(text, str):
        return []
        
    text = text.replace("\r\n", "\n").replace("<|endoftext|>", "")
    _, separator, poem = text.partition("\n\n")
    if not separator:
        poem = text
    return [line for line in poem.split("\n") if line.strip()]

ends, val_words = Counter(), set()  # val_words is for the richness test later
for text in val["text"]:
    for line in poem_lines(text):
        words = words_of(line)
        val_words.update(words)
        if words and not re.fullmatch(r"[ivxlc]+", words[-1]):
            ends[words[-1]] += 1

missing = Counter({w: n for w, n in ends.items() if not phones(w)})
print(f"{sum(missing.values()) / sum(ends.values()):.1%} of line ends unknown")
print(missing.most_common(30))

5.5% of line ends unknown
[('seluen', 25), ('alway', 22), ('paracelsus', 21), ('dere', 21), ('aboute', 19), ('splendour', 18), ('pimlico', 18), ('oute', 18), ('alle', 17), ('lorde', 17), ('ryche', 16), ('haue', 15), ('gawaine', 15), ('bylyue', 15), ('myt', 15), ('palamon', 14), ('carduel', 14), ('wolde', 14), ('clene', 14), ('forbear', 13), ('ope', 13), ('idonea', 13), ('sone', 13), ('mede', 12), ('fayre', 12), ('reposed', 12), ('undefiled', 12), ('seche', 12), ('whyle', 12), ('defiled', 11)]


## 2. Syllables
The number of vowels in the CMUdict pronunciation (they're the phones with a stress digit).
For unknown words we guess from the vowel groups in the spelling.

In [28]:
def guess_syllables(word):
    word = word.replace("'", "")
    n = len(re.findall(r"[aeiouy]+", word))
    if word.endswith("e") and not word.endswith(("le", "ee")) and n > 1:  # silent e
        n -= 1
    return max(n, 1)


def syllables(word):
    p = phones(word)
    if p:
        return len(re.findall(r"\d", p[0]))
    return guess_syllables(word)


def line_syllables(line):
    return sum(syllables(w) for w in words_of(line))


for w in ["light", "heaven", "heav'n", "o'er", "crown'd", "beautiful", "fire", "wondrous", "glimmereth"]:
    print(w, syllables(w))

print(line_syllables("Because I could not stop for Death"))  # 8
print(line_syllables("He kindly stopped for me"))  # 6
print(line_syllables("Of Man's first disobedience, and the fruit"))  # 11, Milton says dis-o-bed-yence so 10

light 1
heaven 2
heav'n 1
o'er 1
crown'd 1
beautiful 3
fire 2
wondrous 2
glimmereth 3
8
6
11


## 3. Rhyme score
The rhyming part of a word is everything from its last stressed vowel to the end (`light` -> `AY1 T`).

| | score | example |
| - | - | - |
| perfect rhyme | 1.0 | night / light |
| same sounds, different stress | 0.9 | sea / melody |
| same vowel (assonance) | 0.6 | light / time |
| same consonants after the vowel (consonance) | 0.3 | light / sat |
| nothing | 0 | |

Rhyming a word with itself scores 0 (night / tonight, light / delight: one is the end of the other and
the sound before the rhyme is the same too), so the model can't rhyme by just repeating itself.
Words that only sound the same do count (sea / see, rain / reign), old poets used those.

In [29]:
def rhyming_part(p):
    p = p.split()
    vowels = [i for i, x in enumerate(p) if x[-1].isdigit()] or [0]  # "hmm" has no vowel
    i = ([v for v in vowels if p[v][-1] in "12"] or vowels)[-1]  # last stressed vowel
    prev = max([v for v in vowels if v < i], default=-1)
    # the rhyming part and the consonants just before it (L in light, K R in crowned)
    return tuple(p[i:]), tuple(p[prev + 1:i])


def no_stress(p):
    return [re.sub(r"\d", "", x) for x in p]


def rhyme_phones(a, b, same_ending):
    ra, before_a = rhyming_part(a)
    rb, before_b = rhyming_part(b)
    if ra == rb:
        return 0 if same_ending and before_a == before_b else 1.0
    # sea / melody: one rhyming part is the end of the other word, if you ignore stress
    # (but not be / maybe, that's the same word again)
    for x, before_x, y in [(ra, before_a, b.split()), (rb, before_b, a.split())]:
        n = len(x)
        if len(y) > n and no_stress(x) == no_stress(y[-n:]):
            return 0 if same_ending and before_x[-1:] == tuple(y[-n - 1:-n]) else 0.9
    if no_stress(ra[:1]) == no_stress(rb[:1]):
        return 0.6
    if len(ra) > 1 and no_stress(ra[1:]) == no_stress(rb[1:]):
        return 0.3
    return 0


def rhyme(w1, w2):
    w1, w2 = w1.lower(), w2.lower()
    if not phones(w1) or not phones(w2):
        return 0
    same_ending = w1.endswith(w2) or w2.endswith(w1)  # light / delight, but also art / heart
    return max(rhyme_phones(a, b, same_ending) for a in phones(w1) for b in phones(w2))


pairs = [("night", "light"), ("sea", "melody"), ("light", "time"), ("light", "sat"), ("light", "dog"),
         ("light", "delight"), ("night", "tonight"), ("love", "prove"), ("love", "dove"), ("o'er", "door"),
         ("wind", "find"), ("eye", "high"), ("crown'd", "ground"), ("thee", "liberty"), ("be", "maybe"),
         ("sea", "see"), ("rain", "reign"), ("art", "heart"), ("light", "light")]
for a, b in pairs:
    print(f"{a:>8} / {b:<8} {rhyme(a, b)}")

   night / light    1.0
     sea / melody   0.9
   light / time     0.6
   light / sat      0.3
   light / dog      0
   light / delight  0
   night / tonight  0
    love / prove    0.3
    love / dove     1.0
    o'er / door     1.0
    wind / find     1.0
     eye / high     1.0
 crown'd / ground   1.0
    thee / liberty  0.9
      be / maybe    0
     sea / see      1.0
    rain / reign    1.0
     art / heart    1.0
   light / light    0


love / prove only scores 0.3: they rhymed for Keats but not in modern American English, which is what CMUdict is.

## 4. Rhyme richness
How many other words rhyme perfectly with a word (idea from PeRDict). Used when a poem opens a new rhyme,
so it doesn't start with a word like *orange* that nothing rhymes with. Identity rhymes (light / delight)
don't count, sound-alikes (sea / see) do.

The real word list comes later (the meaning head's vocabulary), here it's tried on the words in the val poems.

In [30]:
def richness(words):
    keys = {w: {rhyming_part(p) for p in phones(w)} for w in set(words) if phones(w)}
    group = Counter(part for k in keys.values() for part, before in k)
    same_sound = defaultdict(set)
    for w, k in keys.items():
        for key in k:
            same_sound[key].add(w)
    # everything with the same rhyming part, minus the word itself and ones like light / delight
    return {w: max(group[part] - sum(w.endswith(v) or v.endswith(w) for v in same_sound[part, before])
                   for part, before in k) for w, k in keys.items()}


rich = richness(val_words)
print(len(rich), "words")
for w in ["light", "sea", "love", "silver", "month", "orange"]:
    print(w, rich.get(w))

30282 words
light 56
sea 128
love 6
silver 0
month 0
orange 0


## 5. Check on real poems
Share of lines whose last word rhymes perfectly with one of the 2 lines before or after it (catches AABB, ABAB and ABCB).
Pope writes in rhyming couplets and Milton mostly doesn't rhyme, so Pope should come out a lot higher.

In [31]:
train = pd.read_parquet(DATA + "train_test/train.parquet", columns=["author", "text"])


def rhymed_lines(poet):
    rhymed = []
    for text in train[train["author"] == poet]["text"]:
        last = [words_of(line)[-1] for line in poem_lines(text) if words_of(line)]
        for i in range(len(last)):
            near = last[max(i - 2, 0):i] + last[i + 1:i + 3]
            rhymed.append(any(rhyme(last[i], w) == 1 for w in near))
    return pd.Series(rhymed)


for poet in ["Alexander Pope", "John Milton", "Emily Dickinson", "Walt Whitman"]:
    r = rhymed_lines(poet)
    print(f"{poet:<16} {len(r):>6} lines, {r.mean():.0%} rhyme")

Alexander Pope    24036 lines, 70% rhyme
John Milton       17554 lines, 12% rhyme
Emily Dickinson    5084 lines, 28% rhyme
Walt Whitman       9894 lines, 2% rhyme
